In [2]:
import os
import json
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import PowerTransformer

In [3]:
# =========================
# Config
# =========================
DATA_DIR = "ori_data"
OUT_DIR = "ori_data"   # 你也可以改成 "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

PRICE_PATH = os.path.join(DATA_DIR, "price_train.csv")
RENT_PATH  = os.path.join(DATA_DIR, "rent_train.csv")

PRICE_COL = "Price"
RENT_COL  = "Rent"

In [4]:
# =========================
# Helpers
# =========================
def boxcox_transform_mle(y: np.ndarray):
    """
    Box-Cox MLE: 返回 (y_trans, lambda_hat)
    要求 y > 0
    """
    if np.any(~np.isfinite(y)):
        raise ValueError("y contains NaN/inf, please clean first.")
    if np.any(y <= 0):
        raise ValueError("Box-Cox requires y > 0. Found non-positive values.")
    y_trans, lam = stats.boxcox(y)  # MLE lambda
    return y_trans, float(lam)

from sklearn.preprocessing import PowerTransformer
import numpy as np
import pandas as pd

def yeojohnson_transform_mle(y: np.ndarray):
    # ---- 诊断：变换前先看输入是否正常 ----
    s = pd.Series(y)
    print("[YJ] input nunique:", s.nunique(dropna=False),
          "min:", np.nanmin(y), "max:", np.nanmax(y),
          "nan_ratio:", np.mean(~np.isfinite(y)))

    if np.any(~np.isfinite(y)):
        raise ValueError("y contains NaN/inf after numeric conversion. Clean/parse first.")
    if pd.Series(y).nunique() <= 1:
        raise ValueError("y is constant (only one unique value). PowerTransformer will output constant values.")

    pt = PowerTransformer(method="yeo-johnson", standardize=False)
    y2 = y.reshape(-1, 1)
    y_trans = pt.fit_transform(y2).ravel()
    lam = float(pt.lambdas_[0])

    # ---- 诊断：变换后看看是否被压成常数 ----
    st = pd.Series(y_trans)
    print("[YJ] output nunique:", st.nunique(dropna=False),
          "min:", np.min(y_trans), "max:", np.max(y_trans))

    return y_trans, lam

def apply_and_save(df: pd.DataFrame, target_col: str, transform_fn, out_path: str):
    df_out = df.copy()

    # ✅ 更稳：强制按数值解析，并检查异常
    y = pd.to_numeric(df_out[target_col], errors="coerce").to_numpy(dtype=float)

    if np.mean(~np.isfinite(y)) > 0:
        # 这里你可以选择：drop NaN 或者报错
        raise ValueError(f"{target_col} has NaN after parsing. Please inspect raw values (commas/strings/etc.).")

    y_trans, lam = transform_fn(y)
    df_out[target_col] = y_trans
    df_out.to_csv(out_path, index=False)
    return lam


In [5]:
# =========================
# Main
# =========================
price_train = pd.read_csv(PRICE_PATH)
rent_train  = pd.read_csv(RENT_PATH)

assert PRICE_COL in price_train.columns, f"{PRICE_PATH} must contain column '{PRICE_COL}'"
assert RENT_COL in rent_train.columns,   f"{RENT_PATH} must contain column '{RENT_COL}'"

In [6]:
# 1) Box-Cox
price_boxcox_path = os.path.join(OUT_DIR, "price_train_boxcox.csv")
rent_boxcox_path  = os.path.join(OUT_DIR, "rent_train_boxcox.csv")

lam_price_boxcox = apply_and_save(price_train, PRICE_COL, boxcox_transform_mle, price_boxcox_path)
lam_rent_boxcox  = apply_and_save(rent_train,  RENT_COL,  boxcox_transform_mle, rent_boxcox_path)

In [7]:

# 2) Yeo–Johnson
price_yj_path = os.path.join(OUT_DIR, "price_train_yeojohnson.csv")
rent_yj_path  = os.path.join(OUT_DIR, "rent_train_yeojohnson.csv")

lam_price_yj = apply_and_save(price_train, PRICE_COL, yeojohnson_transform_mle, price_yj_path)
lam_rent_yj  = apply_and_save(rent_train,  RENT_COL,  yeojohnson_transform_mle, rent_yj_path)

/root/miniconda3/envs/myconda/lib/python3.8/site-packages/sklearn/preprocessing/_data.py:3253: RuntimeWarning: divide by zero encountered in log
  loglike = -n_samples / 2 * np.log(x_trans.var())
/root/miniconda3/envs/myconda/lib/python3.8/site-packages/sklearn/preprocessing/_data.py:3253: RuntimeWarning: divide by zero encountered in log
  loglike = -n_samples / 2 * np.log(x_trans.var())


In [8]:
# Save lambdas for record
lambda_record = {
    "BoxCox": {
        "Price_lambda": lam_price_boxcox,
        "Rent_lambda": lam_rent_boxcox,
    },
    "YeoJohnson": {
        "Price_lambda": lam_price_yj,
        "Rent_lambda": lam_rent_yj,
    }
}

In [9]:
lambda_path = os.path.join(OUT_DIR, "lambdas_mle.json")
with open(lambda_path, "w", encoding="utf-8") as f:
    json.dump(lambda_record, f, ensure_ascii=False, indent=2)

In [10]:
print("✅ Done.")
print("Box-Cox lambdas:", lambda_record["BoxCox"])
print("Yeo–Johnson lambdas:", lambda_record["YeoJohnson"])
print("Saved files:")
print(" -", price_boxcox_path)
print(" -", rent_boxcox_path)
print(" -", price_yj_path)
print(" -", rent_yj_path)
print("Lambda record saved to:", lambda_path)

✅ Done.
Box-Cox lambdas: {'Price_lambda': -0.0931691824268184, 'Rent_lambda': -0.1585195198596779}
Yeo–Johnson lambdas: {'Price_lambda': -4.472154114139801, 'Rent_lambda': -8.351699460168112}
Saved files:
 - ori_data/price_train_boxcox.csv
 - ori_data/rent_train_boxcox.csv
 - ori_data/price_train_yeojohnson.csv
 - ori_data/rent_train_yeojohnson.csv
Lambda record saved to: ori_data/lambdas_mle.json
